# Hugging Face Accelerate

A comprehensive guide to Hugging Face Accelerate for AI/ML workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

🤗 Accelerate is a lightweight library from Hugging Face that makes it easy to run your existing PyTorch training code on **multiple GPUs, multiple machines, TPUs, or mixed-precision setups** with minimal changes.

### What is it?

At a high level, **Accelerate**:

- Provides an `Accelerator` class that handles **device placement**, **distributed initialization**, and **mixed precision**.
- Wraps your model, optimizer, dataloaders, and scheduler via `accelerator.prepare(...)`.
- Exposes helpers like `accelerator.backward()`, `accelerator.gather()`, and `accelerator.print()` that work correctly in distributed setups.

### Why use it?

Key benefits of using Accelerate:

- **Minimal code changes**: You adapt your existing PyTorch loop instead of rewriting it around a new framework.
- **Backend-agnostic**: Same training loop can run on single-GPU, multi-GPU, multi-node, or TPU environments.
- **Config-driven**: Use `accelerate config` / `accelerate launch` to define distributed and mixed-precision settings without hardcoding them.

### When to use it?

Accelerate is particularly useful when:

- You have a **custom PyTorch training loop** and want to scale it with minimal friction.
- You are using **Hugging Face Transformers** and want a flexible alternative to the higher-level `Trainer`.
- You need to **switch between CPU, GPU, multi-GPU, and TPU** with as little code churn as possible.

## Key Features

### Core Capabilities of Hugging Face Accelerate

| Feature | Description | Benefit |
|--------|-------------|---------|
| **`Accelerator` abstraction** | Single object that configures device, distributed backend, and mixed precision. | One API for many hardware setups. |
| **`prepare(...)` helper** | Wraps model, optimizer, dataloaders, and scheduler. | Automatically handles DDP/FSDP/TPU plumbing. |
| **Mixed precision support** | FP16/BF16 training with automatic scaling. | Faster training and reduced memory usage. |
| **Multi-node / multi-GPU** | Works with PyTorch distributed backends under the hood. | Scale to clusters with little code change. |
| **Integration with Transformers** | Used under the hood by many Hugging Face examples. | Easier scaling of NLP/vision models from the Hub. |
| **Configuration CLI** | `accelerate config` and `accelerate launch` for config-driven runs. | No need to hardcode distributed/mixed-precision details. |

## Architecture Overview

Accelerate is a thin layer around **PyTorch Distributed** (and other backends), orchestrated through the `Accelerator` object.

```text
+----------------------------+
|   Your training script     |
| (PyTorch model + loop)     |
+--------------+-------------+
               |
               |  Accelerator()
               v
+----------------------------+
|        Accelerator         |
|  • Device placement        |
|  • DDP/FSDP setup          |
|  • Mixed precision         |
|  • Logging/printing        |
+--------------+-------------+
               |
               |  prepare(model, optimizer, dataloader, ...)
               v
+----------------------------+
|  Distributed / parallel    |
|  PyTorch under the hood    |
+----------------------------+
```

### Key components

1. **Your training script**  
   - Defines the model, optimizer, dataloaders, scheduler, and training loop.

2. **Accelerator**  
   - Knows how many processes/GPUs there are and which backend to use.  
   - Wraps objects so they are correctly placed and synchronized.

3. **Backends & strategies**  
   - Uses PyTorch’s `torch.distributed` under the hood (DDP, FSDP, etc.).  
   - Can integrate with DeepSpeed, FullyShardedDataParallel, and TPUs through configuration.

## Installation

### Prerequisites

- Python 3.8+.
- PyTorch installed (CPU or GPU build).
- Optional: CUDA-enabled GPUs and drivers for GPU training.

### Install Accelerate

```bash
pip install accelerate
```

You can then run:

```bash
accelerate config
```

to guide you through setting up a default distributed/mixed-precision configuration for your environment.

In [ ]:
# Quick install helper for notebooks (uncomment to run)
# !pip install accelerate torch torchvision

## Basic Usage

### Quick start: wrapping a simple PyTorch loop

The basic pattern is:

1. Create an `Accelerator` instance.  
2. Define your model, optimizer, dataloader, and (optional) scheduler.  
3. Call `accelerator.prepare(...)` on these objects.  
4. Replace `loss.backward()` with `accelerator.backward(loss)`.  
5. Use `accelerator.print()` instead of `print()` for rank-aware logging.

Below is a minimal example for a single-node, multi-GPU setup.

In [ ]:
# Minimal Accelerate + PyTorch example (conceptual)

from accelerate import Accelerator
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


# 1. Create accelerator
accelerator = Accelerator()


# 2. Define model, optimizer, and data
model = nn.Linear(10, 1)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

x = torch.randn(256, 10)
y = torch.randn(256, 1)
train_ds = TensorDataset(x, y)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)


# 3. Prepare objects with accelerator
model, optimizer, train_dl = accelerator.prepare(model, optimizer, train_dl)


# 4. Training loop
for epoch in range(3):
    for batch in train_dl:
        xb, yb = batch
        preds = model(xb)
        loss = ((preds - yb) ** 2).mean()

        optimizer.zero_grad()
        accelerator.backward(loss)
        optimizer.step()

    accelerator.print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

# Note: When launched with `accelerate launch`, this scales to multiple GPUs/nodes.

## Advanced Features

- **Gradient accumulation:** Configure accumulation steps so that each update corresponds to a larger effective batch size.
- **DeepSpeed / FSDP integration:** Use configuration files to enable DeepSpeed ZeRO or PyTorch FSDP under the hood while keeping the same Python training loop.
- **Mixed precision & automatic casting:** Easily enable FP16/BF16 for faster training.  
- **Multi-host (multi-node) setups:** `accelerate launch` handles distributed environment variables for you.
- **Integration with Hugging Face Transformers:** Many official examples use Accelerate to scale Transformer training and inference.

In [ ]:
# Sketch: enabling gradient accumulation with Accelerator (conceptual)

from accelerate import Accelerator

accelerator = Accelerator(gradient_accumulation_steps=4)

# Inside training loop:
# for step, batch in enumerate(train_dl):
#     with accelerator.accumulate(model):
#         outputs = model(batch[0])
#         loss = loss_fn(outputs, batch[1])
#         accelerator.backward(loss)
#         optimizer.step()
#         optimizer.zero_grad()

print("Accelerator can handle gradient accumulation via accumulate().")

## Use Cases

- Scaling **custom PyTorch training loops** from single-GPU to multi-GPU/multi-node.  
- Training Hugging Face Transformers models with custom data pipelines or loss functions.  
- Running mixed-precision training without deep changes to the loop.  
- Prototyping distributed training setups before moving to more specialized frameworks if necessary.

## Best Practices

1. **Start with CPU / single-GPU** using the same Accelerate-powered script, then scale out.  
2. **Use `accelerate config`** to generate configuration files for different environments (local dev, single node with many GPUs, multi-node cluster).  
3. **Leverage `accelerator.print` and `accelerator.log`** instead of raw `print` to avoid duplicated logs.  
4. **Integrate with experiment tracking** (Weights & Biases, MLflow) via the appropriate callbacks/utilities in your loop.  
5. **Keep the training loop clean and framework-idiomatic**; Accelerate is designed to minimize intrusion.

## Common Pitfalls

1. **Forgetting to use `accelerator.prepare`**  
   - Symptom: Model remains on CPU or only one GPU is used.  
   - Fix: Ensure model, optimizer, dataloaders, and scheduler are all passed through `prepare()`.

2. **Mixing raw PyTorch DDP with Accelerate**  
   - Symptom: Confusing or conflicting distributed state.  
   - Fix: Let Accelerate handle distributed setup instead of manually calling `init_process_group()`.

3. **Incorrect mixed-precision usage**  
   - Symptom: NaNs or unstable training when enabling FP16.  
   - Fix: Start with default Accelerate settings, then adjust only if needed.

4. **Launching without `accelerate launch` in multi-process setups**  
   - Symptom: Only one process runs or environment variables are missing.  
   - Fix: Use `accelerate launch your_script.py` for multi-process/multi-node runs.

## Performance Optimization

- **Combine mixed precision + multi-GPU** to get the most out of modern hardware.  
- **Tune batch size and gradient accumulation** to keep GPUs well utilized without OOMs.  
- **Profile end-to-end throughput** (samples/sec, tokens/sec) and step time.  
- For large models, consider **DeepSpeed/FSDP integrations** via Accelerate configs.

In [ ]:
# Placeholder for custom performance benchmarking

print("Measure throughput and step time with and without Accelerate to quantify benefits.")

## Production Deployment

- **Single-node**: Use `accelerate launch` with `--num_processes` equal to the number of GPUs.  
- **Multi-node clusters**: Configure `machine_rank`, `num_machines`, and networking options via Accelerate’s config files or CLI arguments.  
- **Kubernetes / batch systems**: Wrap `accelerate launch` in your job specs (e.g., Kubernetes Jobs, Slurm, Ray Jobs).  
- Store checkpoints in object storage or networked filesystems for later evaluation or serving.

## Monitoring and Observability

- Use **experiment trackers** (Weights & Biases, MLflow, TensorBoard) to log metrics, hyperparameters, and artifacts.  
- Monitor **GPU utilization, memory, and step time** with your usual tooling (e.g., `nvidia-smi`, Prometheus, Grafana).  
- Use `accelerator.print` for rank-aware logging; avoid printing from all processes.
- In multi-node setups, ensure centralized logging or log shipping to simplify debugging.

## Troubleshooting

- **Script runs only on one GPU**: Check that you launched with `accelerate launch` and not plain `python`.  
- **Distributed initialization failures**: Validate your Accelerate config, backend, and network connectivity in multi-node environments.  
- **NaNs or loss spikes when enabling mixed precision**: Temporarily disable mixed precision, confirm stability, then re-enable with default settings.  
- **Inconsistent behavior across runs**: Ensure that seeds are set consistently in all processes when reproducibility is required.

## Comparison with Alternatives

| Aspect | Accelerate | PyTorch DDP / FSDP (raw) | DeepSpeed | Ray Train |
|--------|-----------|--------------------------|----------|----------|
| Abstraction level | Thin wrapper around your loop | Low-level, verbose | Optimization-focused | Orchestration-focused |
| Code changes | Minimal | Larger | Moderate | Moderate |
| Mixed precision | Built-in, simple | Manual AMP setup | Yes | Depends on underlying framework |
| Multi-framework | Primarily PyTorch | PyTorch only | PyTorch | Any (via Ray), but examples focus on PyTorch |

Use Accelerate when you want **simple scaling for PyTorch code with minimal intrusion** and good interoperability with the Hugging Face ecosystem.

## Resources

- Accelerate docs: https://huggingface.co/docs/accelerate  
- Migration tutorial (add Accelerate to your code): https://huggingface.co/docs/accelerate/en/basic_tutorials/migration  
- Blog post introducing Accelerate: https://huggingface.co/blog/accelerate-library

Explore the official examples in the docs and Hugging Face repositories to see more complex training setups (Transformers, DeepSpeed/FSDP integrations, TPUs, etc.).